# BPE from scratch — Exercise

Companion to [Tokenization § Byte-Pair Encoding](https://tanulsingh.github.io/blog/tokenization#byte-pair-encoding-bpe).

You'll build the original BPE algorithm: start with characters, repeatedly merge the most frequent adjacent pair until the vocabulary reaches a target size, then learn how to encode new text using the learned merges.

This is the algorithm behind GPT-2, GPT-3, GPT-4, Llama, and most modern LLMs. The byte-level extension (notebook 02) and PMI variant (WordPiece, notebook 03) are tweaks on top of this core idea.

## Setup

In [ ]:
# TODO: imports — collections.Counter, typing helpers if you want them

## TODO 1 — Pre-tokenize and build initial word frequencies

The efficient way to run BPE: don't process the whole flat token sequence on every iteration. Instead, group identical words together with their counts, and operate on that compact representation.

Take a corpus (list of strings) and produce a dict mapping each unique "word" to its frequency. Each word is represented as a tuple of single characters with a special end-of-word marker `'</w>'` appended.

**Example:**

```
"the cat sat the cat" → {
    ('t', 'h', 'e', '</w>'): 2,
    ('c', 'a', 't', '</w>'): 2,
    ('s', 'a', 't', '</w>'): 1,
}
```

The end-of-word marker is what lets the tokenizer recover word boundaries when encoding new text — without it, you couldn't tell `"theca"` apart from `"the" + "ca"`.

In [ ]:
def pre_tokenize(corpus: list[str]) -> dict[tuple[str, ...], int]:
    """Build {word_as_char_tuple: frequency} from a corpus.

    Steps:
        1. Lowercase + split each string on whitespace
        2. Convert each word to a tuple of single chars + '</w>' marker at the end
        3. Count frequencies across the corpus
    """
    # TODO 1: implement
    pass

In [ ]:
# Sanity check
corpus = ["the cat sat", "the cat ate"]
word_freqs = pre_tokenize(corpus)
for word, freq in word_freqs.items():
    print(word, '→', freq)
# Expected:
#   ('t', 'h', 'e', '</w>') → 2
#   ('c', 'a', 't', '</w>') → 2
#   ('s', 'a', 't', '</w>') → 1
#   ('a', 't', 'e', '</w>') → 1

## TODO 2 — Count adjacent pairs

Given the `word_freqs` dict, count how often each adjacent pair appears across the corpus, weighted by word frequency.

If the word `('c','a','t','</w>')` appears 5 times, then the pairs `('c','a')`, `('a','t')`, `('t','</w>')` each contribute +5 to their counts.

Return a dict mapping `(token_a, token_b) → count`.

In [ ]:
def get_pair_counts(word_freqs: dict[tuple[str, ...], int]) -> dict[tuple[str, str], int]:
    """Count adjacent pairs in the corpus, weighted by word frequency."""
    # TODO 2: implement
    # Hint: collections.Counter; iterate (word, freq) pairs and adjacent token pairs within each word
    pass

In [ ]:
# Sanity check
counts = get_pair_counts(word_freqs)
for pair, count in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    print(pair, '→', count)
# Expected (top entries):
#   ('a', 't') → 3   # 'cat' freq 2 + 'sat' freq 1
#   ('t', 'h') → 2   # 'the' freq 2
#   ...etc

## TODO 3 — Apply a merge

Given a target pair `(a, b)`, walk through every word in `word_freqs` and replace adjacent occurrences of `a` followed by `b` with a single token `"a" + "b"`.

**Example:** if `pair = ('a', 't')`, then `('c','a','t','</w>')` becomes `('c', 'at', '</w>')`.

Return the updated `word_freqs` dict (frequencies stay the same; only the keys change).

In [ ]:
def merge_pair(
    pair: tuple[str, str],
    word_freqs: dict[tuple[str, ...], int],
) -> dict[tuple[str, ...], int]:
    """Replace adjacent `pair` occurrences with the concatenated token in every word."""
    # TODO 3: implement
    # Hint: for each word, build a new tuple by scanning left-to-right and merging matches
    pass

In [ ]:
# Sanity check
merged = merge_pair(('a', 't'), word_freqs)
for word, freq in merged.items():
    print(word, '→', freq)
# Expected:
#   ('t', 'h', 'e', '</w>') → 2     # unchanged (no 'a','t' pair)
#   ('c', 'at', '</w>') → 2          # 'a','t' merged
#   ('s', 'at', '</w>') → 1          # 'a','t' merged
#   ('a', 't', 'e', '</w>') → 1      # NOT merged — 'a','t' aren't adjacent here? Actually they are, this should merge to ('at', 'e', '</w>')

## TODO 4 — The full BPE training loop

Now assemble TODOs 1-3 into a `BPETokenizer` class with a `train` method that:

1. Calls `pre_tokenize` on the corpus to get `word_freqs`
2. Initializes `self.vocab` with all unique single-character tokens
3. Loops until `len(self.vocab) <= vocab_size`:
   - Calls `get_pair_counts(word_freqs)`
   - Picks the pair with maximum count
   - Records the merge in `self.merges` (a list of merge pairs in order)
   - Calls `merge_pair` to update `word_freqs`
   - Adds the new merged token (concatenation) to `self.vocab`
4. (Stop early if no pairs remain.)

The `self.merges` list is the *training output* — you'll replay it during encoding.

In [ ]:
class BPETokenizer:
    """Byte Pair Encoding tokenizer trained from scratch."""

    def __init__(self):
        self.merges: list[tuple[str, str]] = []   # learned merges, in order
        self.vocab: set[str] = set()              # final vocabulary

    def train(self, corpus: list[str], vocab_size: int):
        """Learn BPE merges until the vocabulary reaches `vocab_size`."""
        pass

    def encode(self, text: str) -> list[str]:
        # implemented in TODO 5
        raise NotImplementedError("TODO 5")

In [ ]:
# Sanity check — train on a tiny corpus
corpus = ["the cat sat", "the cat ate", "the bat sat"] * 5
tok = BPETokenizer()
tok.train(corpus, vocab_size=20)
print('vocab size:', len(tok.vocab))
print('vocab:', sorted(tok.vocab))
print('first few merges:', tok.merges[:5])
# You should see merges that build up common substrings like 't','h' → 'th', then 'th','e' → 'the', etc.

## TODO 5 — Encode new text using the learned merges

At inference, you don't recount pairs — you just **replay the learned merges in order**.

Algorithm:
1. Pre-tokenize the input text the same way (split on whitespace, add `'</w>'`, tuple of chars)
2. For each word, walk through `self.merges` in order. For each merge `(a, b)`, scan the word and replace any adjacent `(a, b)` pair with the concatenated token.
3. Concatenate all word token lists into a single flat list. Return.

Go back to your `BPETokenizer` class and replace the `encode` method's `raise NotImplementedError` with the real implementation.

In [ ]:
# Sanity check encoding
tokens = tok.encode("the cat")
print(tokens)
# After enough merges, you should see something like ['the</w>', 'cat</w>'] or close.
# Without merges, you'd see individual chars: ['t','h','e','</w>','c','a','t','</w>']

## Run the tests

In [ ]:
from tests import run_bpe_tests
# run_bpe_tests(BPETokenizer, pre_tokenize, get_pair_counts, merge_pair)